# 01 — Data Exploration

**Purpose:** Download all 18 data series (17 from FRED + 1 from WID.world), verify availability, and visually inspect the raw data before building models.

**Prerequisites:**
1. `pip install -r requirements.txt`
2. Copy `.env.example` to `.env` and add your FRED API key
   - Get a free key at: https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

from config import FRED_SERIES
from src.data.fred_client import FREDClient
from src.data.wid_loader import WIDLoader
from src.data.data_pipeline import DataPipeline

## Step 1: Download all FRED series

In [ ]:
client = FREDClient()
fred_data = client.get_all_configured_series(use_cache=True)

print(f"\nDownloaded {len(fred_data)}/{len(FRED_SERIES)} series successfully")

In [ ]:
# Show catalog summary
catalog = client.get_catalog_summary()
catalog

## Step 2: Download WID.world top 1% income share

In [ ]:
wid = WIDLoader()
try:
    wid_data = wid.get_top1_income_share()
    print(f"WID data: {len(wid_data)} observations, {wid_data.index.min().date()} to {wid_data.index.max().date()}")
except Exception as e:
    print(f"WID download failed: {e}")
    print("You can manually download from wid.world/data/ and use wid.load_from_csv()")
    wid_data = None

## Step 3: Build unified monthly dataset

In [ ]:
pipeline = DataPipeline()
unified = pipeline.build_unified_monthly(fred_data, wid_data)

print(f"Unified dataset: {unified.shape[1]} columns x {len(unified)} months")
print(f"Date range: {unified.index.min().date()} to {unified.index.max().date()}")
print(f"\nNull counts per series:")
print(unified.isnull().sum().sort_values(ascending=False))

In [ ]:
# Data quality report
quality = pipeline.data_quality_report(unified)
quality

## Step 4: Plot all series by model

### Turchin PSI inputs

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

psi_series = [
    ('W270RE1A156NBEA', 'Labor Share of GDP (%)', 'Inverted for MMP'),
    ('top_1pct_share', 'Top 1% Income Share', 'EMP proxy'),
    ('GFDEGDQ188S', 'Federal Debt / GDP (%)', 'SFD proxy'),
]

for i, (sid, title, subtitle) in enumerate(psi_series):
    if sid in unified.columns:
        data = unified[sid].dropna()
        axes[i].plot(data.index, data.values, linewidth=1.2)
        axes[i].set_title(f'{title} — {subtitle}', fontsize=11)
        axes[i].set_ylabel(title)
        axes[i].grid(alpha=0.3)
    else:
        axes[i].text(0.5, 0.5, f'{sid} not available', transform=axes[i].transAxes,
                    ha='center', va='center', fontsize=14, color='red')

axes[-1].xaxis.set_major_locator(mdates.YearLocator(10))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.suptitle('Turchin PSI — Input Data', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

### Prospect Theory PLI inputs

In [ ]:
pli_series = [
    ('MEHOINUSA672N', 'Real Median Household Income ($)'),
    ('FIXHAI', 'Housing Affordability Index'),
    ('SPDYNLE00INUSA', 'Life Expectancy (years)'),
    ('LNS12300060', 'Prime-Age Employment Ratio (%)'),
    ('UMCSENT', 'Consumer Sentiment Index'),
]

fig, axes = plt.subplots(len(pli_series), 1, figsize=(14, 3 * len(pli_series)), sharex=True)

for i, (sid, title) in enumerate(pli_series):
    if sid in unified.columns:
        data = unified[sid].dropna()
        axes[i].plot(data.index, data.values, linewidth=1.2, color='#d7191c')
        axes[i].set_title(title, fontsize=11)
        axes[i].set_ylabel(sid)
        axes[i].grid(alpha=0.3)
    else:
        axes[i].text(0.5, 0.5, f'{sid} not available', transform=axes[i].transAxes,
                    ha='center', va='center', fontsize=14, color='red')

axes[-1].xaxis.set_major_locator(mdates.YearLocator(10))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.suptitle('Prospect Theory PLI — Input Data', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

### Financial Stress Pathway inputs

In [ ]:
fssi_series = [
    ('STLFSI4', 'St. Louis Fed Financial Stress Index'),
    ('T10Y2Y', '10Y-2Y Treasury Spread (%)'),
    ('VIXCLS', 'VIX Volatility Index'),
    ('BAMLH0A0HYM2', 'High-Yield Credit Spread (%)'),
    ('DRSFRMACBS', 'Mortgage Delinquency Rate (%)'),
]

eti_series = [
    ('UNRATE', 'Unemployment Rate (%)'),
    ('IC4WSA', 'Initial Claims 4-Week MA'),
    ('CSCICP03USM665S', 'Consumer Confidence (OECD)'),
    ('CPIAUCSL', 'CPI (All Urban Consumers)'),
    ('CES0500000003', 'Avg Hourly Earnings ($)'),
]

fig, axes = plt.subplots(5, 2, figsize=(16, 15))

for i, (sid, title) in enumerate(fssi_series):
    ax = axes[i, 0]
    if sid in unified.columns:
        data = unified[sid].dropna()
        ax.plot(data.index, data.values, linewidth=1.0, color='#e41a1c')
        ax.set_title(f'FSSI: {title}', fontsize=10)
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'{sid} N/A', transform=ax.transAxes, ha='center', color='red')

for i, (sid, title) in enumerate(eti_series):
    ax = axes[i, 1]
    if sid in unified.columns:
        data = unified[sid].dropna()
        ax.plot(data.index, data.values, linewidth=1.0, color='#377eb8')
        ax.set_title(f'ETI: {title}', fontsize=10)
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'{sid} N/A', transform=ax.transAxes, ha='center', color='red')

fig.suptitle('Financial Stress Pathway — Input Data (FSSI left, ETI right)',
            fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

## Step 5: Quick model test

Run all three models on the most recent date to verify they compute without errors.

In [ ]:
from src.models.turchin_psi import TurchinPSI
from src.models.prospect_theory import ProspectTheoryPLI
from src.models.financial_stress import FinancialStressPathway

latest_date = unified.index[-1]
print(f"Computing models for: {latest_date.date()}\n")

# Turchin PSI
try:
    psi = TurchinPSI()
    psi_out = psi.compute(unified, latest_date)
    print(f"Turchin PSI: {psi_out.score:.1f}")
    for k, v in psi_out.components.items():
        print(f"  {k}: {v:.1f}")
    if psi_out.flags:
        print(f"  Flags: {psi_out.flags}")
except Exception as e:
    print(f"PSI failed: {e}")

print()

# Prospect Theory PLI
try:
    pli = ProspectTheoryPLI()
    pli_out = pli.compute(unified, latest_date)
    print(f"Prospect Theory PLI: {pli_out.score:.1f}")
    for k, v in pli_out.components.items():
        print(f"  {k}: {v:.1f}")
    if pli_out.flags:
        print(f"  Flags: {pli_out.flags}")
except Exception as e:
    print(f"PLI failed: {e}")

print()

# Financial Stress Pathway
try:
    fsp = FinancialStressPathway()
    fsp_out = fsp.compute(unified, latest_date)
    print(f"Financial Stress Pathway: {fsp_out.score:.1f}")
    print(f"  FSSI (Financial): {fsp_out.components.get('FSSI', 'N/A')}")
    print(f"  ETI (Economic): {fsp_out.components.get('ETI', 'N/A')}")
    if fsp_out.flags:
        print(f"  Flags: {fsp_out.flags}")
except Exception as e:
    print(f"FSP failed: {e}")

## Step 6: Compute full historical series and plot

Run all three models from 1970 to present and produce the multi-panel visualization.

In [ ]:
from src.visualization.plots import plot_multi_panel

model_results = {}

# Turchin PSI historical
try:
    psi_model = TurchinPSI()
    psi_hist = psi_model.compute_historical(unified, start='1970-01-01')
    model_results['turchin_psi'] = psi_hist
    print(f"PSI: {len(psi_hist)} months computed")
except Exception as e:
    print(f"PSI historical failed: {e}")

# PLI historical
try:
    pli_model = ProspectTheoryPLI()
    pli_hist = pli_model.compute_historical(unified, start='1984-01-01')
    model_results['prospect_theory'] = pli_hist
    print(f"PLI: {len(pli_hist)} months computed")
except Exception as e:
    print(f"PLI historical failed: {e}")

# FSP historical
try:
    fsp_model = FinancialStressPathway()
    fsp_hist = fsp_model.compute_historical(unified, start='1996-01-01')
    model_results['financial_stress'] = fsp_hist
    print(f"FSP: {len(fsp_hist)} months computed")
except Exception as e:
    print(f"FSP historical failed: {e}")

In [ ]:
# Compute simple composite where all models overlap
composite = None
if model_results:
    all_scores = pd.DataFrame({
        name: df['score'] for name, df in model_results.items()
    })
    composite = all_scores.mean(axis=1)

fig = plot_multi_panel(model_results, composite=composite)
plt.show()

## Step 7: Quick backtesting check

In [ ]:
from src.analysis.backtesting import Backtester
from src.visualization.plots import plot_backtesting_summary

bt = Backtester(detection_threshold=45)

model_series = {name: df['score'] for name, df in model_results.items()}
report = bt.full_report(model_series)

print("=== BACKTESTING SUMMARY ===")
for model, summary in report['summary'].items():
    print(f"\n{model}:")
    for k, v in summary.items():
        print(f"  {k}: {v}")

print("\n=== INTER-MODEL CORRELATIONS ===")
print(report['correlations'].round(3))

In [ ]:
fig = plot_backtesting_summary(report)
plt.show()

## Next Steps

After reviewing the output:
1. **If PSI flatlines:** The geometric mean fix may not be enough; consider the probability union form
2. **If PLI saturates at 100 during 2008/2020:** K constants still too high; reduce further
3. **If FSSI and ETI don't show lag:** The transmission may be more rapid than hypothesized
4. **If inter-model r > 0.85:** Models are not independent; investigate shared drivers
5. **If models don't detect 2008/2020:** Fundamental model design issue; revisit formulas

See `02_psi_implementation.ipynb` through `05_backtesting.ipynb` for deeper dives into each model.